In [0]:
#place it in a _setup notebook inside the explorations folder

#need to run this start of each cell one time for the notebook to work, dont like having filepaths, so I did this

#import sys

#PROJECT_ROOT = "/Workspace/Users/<email>/<projectname(root)/<pyproject location> if pyproject is in root this last one isnt needed"

#if PROJECT_ROOT not in sys.path:
    #sys.path.insert(0, PROJECT_ROOT)

In [0]:
%run ../explorations/_setup

need to fix warning for dense_rank() but its included for the doc to be used, can cause degradation in too big of a dataset. will have too look into this later

In [0]:
from importlib import reload
import transformations.silver.clean_silver as silver_module
reload(silver_module)

In [0]:
from transformations.silver.clean_silver import build_silver

build_silver(spark)

In [0]:
from pyspark.sql import functions as F
from utils.helpers import get_table
#athlete ID SHOULD be unique, but this is not the case, or the data is kinda dirty where the same athlete is listed multiple times for the same event, so we need to do a join to get the unique athlete ID
df_silver = get_table("marathos.silver.obt_results")

(df_silver.groupBy("athlete_id_raw", "event_id") 
  .count() 
  .filter(F.col("count") > 1) 
  .orderBy(F.desc("count")) 
  .limit(10) 
  .display())


In [0]:
df_silver = get_table("marathos.silver.obt_results")

# Check unit distribution
(df_silver.groupBy("event_distance_unit") 
  .count() 
  .orderBy(F.desc("count")) 
  .display())

In [0]:
df_silver = get_table("marathos.silver.obt_results")
print(f"Silver rows:      {df_silver.count()}")

fct_results = get_table("marathos.gold.fct_results")
print(f"fct_results rows: {fct_results.count()}")

In [0]:
from utils.helpers import get_table
df_silver = get_table("marathos.silver.obt_results")
(df_silver.select("athlete_id_raw", "athlete_id", "athlete_country", "athlete_gender") 
  .limit(10) 
  .display())

In [0]:
(df_silver.select("event_distance_raw", "event_distance_unit") 
  .filter(F.col("event_distance_raw").like("%h%")) 
  .limit(10) 
  .display())

In [0]:
df_silver = get_table("marathos.silver.obt_results")
df_silver.groupBy("event_distance_unit").count().orderBy(F.desc("count")).display()

# verify the output

In [0]:
from utils.helpers import get_table
df_silver = get_table("marathos.silver.obt_results")
print(f"Rows: {df_silver.count()}")
df_silver.printSchema()
df_silver.limit(5).display()

# check the distance units

In [0]:
from pyspark.sql import functions as F
(df_silver.groupBy("event_distance_unit") 
  .count() 
  .orderBy(F.desc("count")) 
  .display())

# verify performance_seconds looks correct

In [0]:
df_silver.select(
    "athlete_performance_raw",
    "performance_seconds"
).limit(10).display()

# check ids were created correctly

In [0]:
(df_silver.select("event_name", "event_id") 
  .distinct() 
  .orderBy("event_id") 
  .limit(10) 
  .display())

In [0]:
df_silver = get_table("marathos.silver.obt_results")
df_silver.printSchema()

In [0]:
print(f"Total rows : {df_silver.count()}")
print(f"Unique events :{df_silver.select('event_id').distinct().count()}")
print(f"Unique athletes :{df_silver.select('athlete_id').distinct().count()}")

For the DBML
these are the names on the dataset website, year of event is questionable. but it would most likely cause duplicates if placed into event and not result
 

Year of event, Event dates, Event name, Event distance/length, 
Event number of finishers, 

Athlete performance, Athlete club, 
Athlete country, Athlete year of birth, Athlete gender, 
Athlete age category, Athlete average speed, Athlete ID

 Athlete is placed into Athlete and event into event. result get the measurable outcomes

In [0]:
df_silver = get_table("marathos.silver.obt_results")

# Check for duplicate rows based on what should be a unique combination
(df_silver.groupBy("event_id", "athlete_id", "year_of_event") 
  .count() 
  .filter(F.col("count") > 1) 
  .orderBy(F.desc("count")) 
  .limit(10) 
  .display())